In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk"
os.environ["SPARK_SUBMIT_OPTS"] = "--add-exports=java.base/sun.nio.ch=ALL-UNNAMED"

In [ ]:
import os
import sys
import pyspark
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk"
print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python: 3.14.7 (main, Aug 10 2026, 07:46:56) [GCC 16.1.1 20260728]
PySpark: 4.2.0
JAVA_HOME: /usr/lib/jvm/java-17-openjdk


In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk"
os.environ["SPARK_SUBMIT_OPTS"] = "--add-exports=java.base/sun.nio.ch=ALL-UNNAMED"

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk"

from pyspark.sql import SparkSession

try:
    spark = (
        SparkSession.builder
        .master("local[1]")
        .appName("test")
        .getOrCreate()
    )
    print("SUCCESS:", spark.version)

except Exception as e:
    import traceback
    traceback.print_exc()

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/vbs2004/Downloads/infra-copmpose-pipeline-general/infra_pipeline (2)/infra_pipeline/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/12 12:57:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Traceback (most recent call last):
  File "/tmp/ipykernel_241774/26876177.py", line 11, in <module>
    .getOrCreate()
     ~~~~~~~~~~~^^
  File "/home/vbs2004/Downloads/infra-copmpose-pipeline-general/infra_pipeline (2)/infra_pipeline/.venv/lib/python3.14/site-packages/pyspark/sql/session.py", line 56

In [ ]:
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("terraformer-v2").getOrCreate()

conn = sqlite3.connect("data/TerraDS.sqlite")
repos_df    = pd.read_sql("SELECT * FROM Repositories", conn)
modules_df  = pd.read_sql("SELECT * FROM Modules", conn)
resources_df = pd.read_sql("SELECT * FROM Resources", conn)

repos_spark    = spark.createDataFrame(repos_df)
modules_spark  = spark.createDataFrame(modules_df)
resources_spark = spark.createDataFrame(resources_df)

print("Raw repos:", repos_spark.count())

TypeError: 'JavaPackage' object is not callable

In [ ]:
filtered_df = repos_spark.filter(
    repos_spark.License.isin(['MIT', 'Apache-2.0', 'BSD-3-Clause'])
)
print("After license filter:", filtered_df.count())   # should be your 50769

26/09/12 12:20:40 WARN TaskSetManager: Stage 6 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


After license filter: 50769


In [ ]:
active_df  = filtered_df.filter(F.col("Archived") == 0)
deduped_df = active_df.dropDuplicates(["FullName"])
sized_df   = deduped_df.filter(F.col("SizeInKb") > 5)

print("Active:", active_df.count())
print("Deduped:", deduped_df.count())
print("Sized (final):", sized_df.count())


26/09/12 12:23:45 WARN TaskSetManager: Stage 10 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


Active: 48857


26/09/12 12:23:46 WARN TaskSetManager: Stage 13 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


Deduped: 48857


26/09/12 12:23:47 WARN TaskSetManager: Stage 19 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


Sized (final): 44188


In [ ]:
modules_renamed = modules_spark.withColumnRenamed("Id", "ModuleId")

repo_modules = modules_renamed.join(
    sized_df.select(F.col("Id").alias("RepoId"), "FullName", "License", "StarCount"),
    modules_renamed.RepositoryId == F.col("RepoId"),
    "inner"
)

resources_renamed = resources_spark.withColumnRenamed("Id", "ResourceId")

full_joined = repo_modules.join(
    resources_renamed,
    repo_modules.ModuleId == resources_renamed.ModuleId,
    "inner"
)

print("Modules in filtered repos:", repo_modules.count())
print("Resources in filtered repos:", full_joined.count())

full_joined.printSchema()

26/09/12 12:26:47 WARN TaskSetManager: Stage 89 contains a task of very large size (3401 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:47 WARN TaskSetManager: Stage 90 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


Modules in filtered repos: 218957


26/09/12 12:26:48 WARN TaskSetManager: Stage 98 contains a task of very large size (3401 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:48 WARN TaskSetManager: Stage 99 contains a task of very large size (9479 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:49 WARN TaskSetManager: Stage 100 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


Resources in filtered repos: 1398461
root
 |-- ModuleId: long (nullable = true)
 |-- RepositoryId: long (nullable = true)
 |-- Path: string (nullable = true)
 |-- Providers: string (nullable = true)
 |-- ModuleCalls: string (nullable = true)
 |-- DiagnosticMessages: string (nullable = true)
 |-- RepoId: long (nullable = true)
 |-- FullName: string (nullable = true)
 |-- License: string (nullable = true)
 |-- StarCount: long (nullable = true)
 |-- ResourceId: long (nullable = true)
 |-- ModuleId: long (nullable = true)
 |-- ResourceType: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Provider: string (nullable = true)



In [ ]:
full_joined_clean = full_joined.drop(resources_renamed.ModuleId)

full_joined_clean.printSchema()   # confirm only ONE ModuleId now, no other dupes

full_joined_clean.groupBy("ResourceType").count().orderBy(F.desc("count")).show(25, truncate=False)
full_joined_clean.groupBy("Provider").count().orderBy(F.desc("count")).show(25, truncate=False)

sized_df.write.mode("overwrite").parquet("data/terrads_filtered.parquet")
full_joined_clean.write.mode("overwrite").parquet("data/terrads_joined.parquet")

root
 |-- ModuleId: long (nullable = true)
 |-- RepositoryId: long (nullable = true)
 |-- Path: string (nullable = true)
 |-- Providers: string (nullable = true)
 |-- ModuleCalls: string (nullable = true)
 |-- DiagnosticMessages: string (nullable = true)
 |-- RepoId: long (nullable = true)
 |-- FullName: string (nullable = true)
 |-- License: string (nullable = true)
 |-- StarCount: long (nullable = true)
 |-- ResourceId: long (nullable = true)
 |-- ResourceType: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Provider: string (nullable = true)



26/09/12 12:26:50 WARN TaskSetManager: Stage 113 contains a task of very large size (9479 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:52 WARN TaskSetManager: Stage 114 contains a task of very large size (3401 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:52 WARN TaskSetManager: Stage 115 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


+------------+-------+
|ResourceType|count  |
+------------+-------+
|Managed     |1176599|
|Data        |221862 |
+------------+-------+



26/09/12 12:26:53 WARN TaskSetManager: Stage 128 contains a task of very large size (3401 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:53 WARN TaskSetManager: Stage 129 contains a task of very large size (9479 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:54 WARN TaskSetManager: Stage 130 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.


+-----------+------+
|Provider   |count |
+-----------+------+
|aws        |715144|
|azurerm    |211932|
|google     |126397|
|random     |34634 |
|kubernetes |28735 |
|null       |24350 |
|keycloak   |17654 |
|template   |15408 |
|local      |14187 |
|ibm        |12225 |
|oci        |12109 |
|tls        |10124 |
|google-beta|9399  |
|terraform  |8939  |
|helm       |8882  |
|alicloud   |7991  |
|openstack  |6505  |
|github     |6334  |
|yandex     |5871  |
|azuread    |5711  |
|vsphere    |5564  |
|vault      |5443  |
|azurecaf   |5420  |
|kubectl    |4765  |
|archive    |4699  |
+-----------+------+
only showing top 25 rows


26/09/12 12:26:55 WARN TaskSetManager: Stage 143 contains a task of very large size (2333 KiB). The maximum recommended task size is 1000 KiB.
26/09/12 12:26:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/12 12:26:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/12 12:26:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/12 12:26:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/09/12 12:26:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/09/12 12:26:56 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,05

[1177.034s][warning][gc,alloc] Executor task launch worker for task 4.0 in stage 156.0 (TID 559): Retried waiting for GCLocker too often allocating 1048578 words


26/09/12 12:27:00 WARN TaskMemoryManager: Failed to allocate a page (8388608 bytes) for 0 times, try again.
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.unsafe.memory.HeapMemoryAllocator.allocate(HeapMemoryAllocator.java:72)
	at org.apache.spark.memory.TaskMemoryManager.allocatePage(TaskMemoryManager.java:398)
	at org.apache.spark.memory.TaskMemoryManager.allocatePage(TaskMemoryManager.java:359)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:97)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.growPointerArrayIfNecessary(UnsafeExternalSorter.java:419)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.allocateMemoryForRecordIfNecessary(UnsafeExternalSorter.java:476)
	at org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter.insertRecord(UnsafeExternalSorter.java:527)
	at org.apache.spark.sql.execution.UnsafeExternalRowSorter.insertRow(UnsafeExternalRowSorter.java:141)
	at org.apache.spa

In [ ]:
# ============================================
# Sanity check: does the tar.gz name match a real Id?
# ============================================
import tarfile

# Pick one archive to inspect — replace with a real filename you have
sample_path = "data/TerraDS_CodeRepos/1067203.tar.gz"

with tarfile.open(sample_path, "r:gz") as tar:
    members = tar.getnames()
    print(f"Total files/dirs inside: {len(members)}")
    for name in members[:20]:
        print(name)

Total files/dirs inside: 12
1067203
1067203/conf
1067203/conf/ci
1067203/conf/ci/backend-feedpaper-api.tf
1067203/conf/ci/backend-feedpaper-data.tf
1067203/conf/ci/backend-feedpaper-web.tf
1067203/feedpaper-api
1067203/feedpaper-api/infrastructure.tf
1067203/feedpaper-data
1067203/feedpaper-data/infrastructure.tf
1067203/feedpaper-web
1067203/feedpaper-web/infrastructure.tf


In [ ]:
match = repos_df[repos_df["Id"] == 1067203]
print(match[["Id", "FullName", "License"]] if not match.empty else "No match found for that Id")

NameError: name 'repos_df' is not defined